In [1]:
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import Row


StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 3, Finished, Available, Finished, False)

In [2]:
# Generate a unique run identifier using the given prefix and a UUID
def new_dq_run_id(prefix="dq"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    unique = uuid.uuid4().hex[:8]
    return f"{prefix}_{timestamp}_{unique}"

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 4, Finished, Available, Finished, False)

In [3]:
# Build a standardized data quality result row with rule metadata, metric values, and evaluation status

def make_rule_result_row(
    dq_run_id,
    pipeline_run_id,
    run_ts,
    layer,
    table_name,
    rule_id,
    rule_name,
    rule_type,
    column_name,
    severity,
    dimension,
    total_count,
    failed_count,
    failed_rate,
    threshold_rate,
    status,
    rule_message
):
    return {
        "dq_run_id": dq_run_id,
        "pipeline_run_id": pipeline_run_id,
        "run_ts": run_ts,
        "layer": layer,
        "table_name": table_name,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "rule_type": rule_type,
        "column_name": column_name,
        "severity": severity,
        "dimension": dimension,
        "total_count": int(total_count) if total_count is not None else None,
        "failed_count": int(failed_count) if failed_count is not None else None,
        "failed_rate": float(failed_rate) if failed_rate is not None else None,
        "threshold_rate": float(threshold_rate) if threshold_rate is not None else None,
        "status": status,
        "rule_message": rule_message
    }

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 5, Finished, Available, Finished, False)

In [4]:
# 这个函数处理大多数规则，比如：空值检查,范围检查,非法值检查

def eval_standard_rule(
    df,
    rule,
    dq_run_id,
    pipeline_run_id,
    layer,
    table_name,
    run_ts
):
    total_count = df.count()

    failed_count = df.filter(rule["fail_cond"]).count()
    failed_rate = (failed_count)/ total_count if total_count > 0 else 0.0
    threshold_rate = rule["threshold_rate"]

    status = "FAIL" if failed_rate > threshold_rate else "PASS"

    rule_message = (
        f"{rule['rule_name']} failed_rate = {failed_rate:.6f}, "
        f"threshold_rate = {threshold_rate:.6f}"
    )

    return make_rule_result_row(
        dq_run_id = dq_run_id,
        pipeline_run_id = pipeline_run_id,
        run_ts = run_ts,
        layer = layer,
        table_name = table_name,
        rule_id = rule["rule_id"],
        rule_name = rule["rule_name"],
        rule_type = rule["rule_type"],
        column_name = rule.get("column_name"),
        severity = rule["severity"],
        dimension = rule["dimension"],
        total_count = total_count,
        failed_count = failed_count,
        failed_rate = failed_rate,
        threshold_rate = threshold_rate,
        status = status,
        rule_message = rule_message
    )
    



StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 6, Finished, Available, Finished, False)

In [5]:
# duplicate rule 执行函数,这个专门处理唯一性检查,因为 duplicate 不能只靠普通 fail_cond。

def eval_duplicate_key_rule(
    df,
    rule,
    dq_run_id,
    pipeline_run_id,
    layer,
    table_name,
    run_ts
):
    key_cols = rule["key_cols"]

    total_count = df.count()

    dup_df = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_rows = dup_df.agg(F.sum("count").alias("dup_row_cnt")).collect()[0]["dup_row_cnt"]
    failed_count = duplicate_rows if duplicate_rows is not None else 0

    failed_rate = (failed_count / total_count) if total_count > 0 else 0.0
    threshold_rate = rule["threshold_rate"]

    status = "FAIL" if failed_rate > threshold_rate else "PASS"

    rule_message = (
        f"{rule['rule_name']} failed_rate={failed_rate:.6f}, "
        f"threshold_rate={threshold_rate:.6f}"
    )

    return make_rule_result_row(
        dq_run_id = dq_run_id,
        pipeline_run_id = pipeline_run_id,
        run_ts = run_ts,
        layer = layer,
        table_name = table_name,
        rule_id = rule["rule_id"],
        rule_name = rule["rule_name"],
        rule_type = rule["rule_type"],
        column_name = ",".join(key_cols),
        severity = rule["severity"],
        dimension = rule["dimension"],
        total_count = total_count,
        failed_count = failed_count,
        failed_rate = failed_rate,
        threshold_rate = threshold_rate,
        status = status,
        rule_message = rule_message
    )


StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 7, Finished, Available, Finished, False)

In [6]:
#生成 gate result, 这个函数根据所有规则结果，给整张表一个最终结论。

def build_gate_result(
    rule_results,
    dq_run_id,
    pipeline_run_id,
    layer,
    table_name,
    run_ts
):
    total_rules = len(rule_results)
    passed_rules = sum(1 for r in rule_results if r["status"] == "PASS")
    failed_rules = sum(1 for r in rule_results if r["status"] == "FAIL")

    critical_failed_rules = sum(
        1 for r in rule_results
        if r["status"] == "FAIL" and r["severity"] == "CRITICAL"
    )

    major_failed_rules = sum(
        1 for r in rule_results
        if r["status"] == "FAIL" and r["severity"] == "MAJOR"
    )

    if critical_failed_rules > 0:
        decision = "BLOCCKED"
        decision_reason = "At least one CRITICAL rule failed"
    elif major_failed_rules > 0:
        decision = "DEGRADED"
        decision_reason = "One or more Major rules failed"
    else:
        decision = "PASS"
        decision_reason = "All rules passed or only non-blocking issues found"
    
    return {
        "dq_run_id": dq_run_id,
        "pipeline_run_id": pipeline_run_id,
        "run_ts": run_ts,
        "layer": layer,
        "table_name": table_name,
        "total_rules": total_rules,
        "passed_rules": passed_rules,
        "failed_rules": failed_rules,
        "critical_failed_rules": critical_failed_rules,
        "major_failed_rules": major_failed_rules,
        "decision": decision,
        "decision_reason": decision_reason
    }

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 8, Finished, Available, Finished, False)

In [7]:
# 新旧版本分界线------

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 9, Finished, Available, Finished, False)

In [8]:
# # Check the total number of rows against a minimum threshold and return a data quality result row
# def dq_row_count(df, run_id, layer, dataset, min_rows=1):
#     cnt = df.count()
#     status = "PASS" if cnt >= min_rows else "FAIL"
#     details = f"row_count={cnt}, min_rows={min_rows}"
#     return [_base_row(run_id, layer, dataset, "row_count", "row_count", None, cnt, min_rows, status, details)]


StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 10, Finished, Available, Finished, False)

In [9]:
# # Compute null ratios for multiple columns using a single aggregation to avoid repeated full-table scans
# def dq_null_ratio(df,run_id, layer, dataset,columns, max_null_ratio=0.05):
#     total = df.count()

#     # Build aggregation expressions for all columns
#     agg_exprs = [
#         F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
#         for c in columns
#     ]

#     null_counts = df.agg(*agg_exprs).collect()[0].asDict()

#     rows = []
#     for c in columns:
#         null_cnt = null_counts[c]
#         ratio = (null_cnt / total) if total > 0 else 0.0
#         status = "PASS" if ratio <= max_null_ratio else "FAIL"
#         details = f"null_count={null_cnt}, total={total},null_ratio={ratio}"

#         rows.append(
#             _base_row(
#                 run_id, layer, dataset,
#                 f"null_ratio: {c}", "null_ratio",c,
#                 ratio, max_null_ratio, status, details
#             )
#         )
        
#     return rows

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 11, Finished, Available, Finished, False)

In [10]:
# # Check whether column values fall within the specified range and return a data quality result row
# def dq_value_range(df,run_id, layer, dataset, column, min_value=None, max_value=None,allowed_null=True):
#     cond = F.lit(True)

#     if min_value is not None:
#             cond = cond & (F.col(column) >= F.lit(min_value))
    
#     if max_value is not None:
#             cond = cond & (F.col(column) <= F.lit(max_value))

#     if allowed_null:
#         cond = cond | F.col(column).isNull()
    
#     bad = df.filter(~cond).count()
#     status = "PASS" if bad == 0 else "FAIL"
#     details = f"bad_rows={bad},min={min_value},max={max_value},allowed_null={allowed_null}"

#     metric_value = bad
#     threshold = 0.0
#     return [_base_row(
#         run_id, layer,dataset,
#         f"value_range:{column}","value_range", 
#         column, metric_value, threshold, 
#         status, details
#     )]


StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 12, Finished, Available, Finished, False)

In [11]:
# # uniqueness check
# def dq_uniqueness(df, run_id, layer, dataset, column):
#     dup_cnt = (
#         df.groupBy(column)
#           .count()
#           .filter(F.col("count") > 1)
#           .count()
#     )

#     status = "PASS" if dup_cnt == 0 else "FAIL"
#     details = f"duplicate_{column}_count={dup_cnt}"

#     return [_base_row(
#         run_id, layer, dataset,
#         f"unique:{column}", "uniqueness",
#         column,
#         dup_cnt, 0.0,
#         status,
#         details
#     )]

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 13, Finished, Available, Finished, False)

In [12]:
# # Convert a list of data quality result rows into a Spark DataFrame with a predefined schema and a timestamp column.
# def dq_to_df(spark,rows):
#     schema = T.StructType([
#         T.StructField("run_id", T.StringType(), False),
#         T.StructField("layer",T.StringType(), False),
#         T.StructField("dataset",T.StringType(), False),
#         T.StructField("rule_name",T.StringType(), False),
#         T.StructField("rule_type",T.StringType(), False),
#         T.StructField("column_name",T.StringType(), True),
#         T.StructField("metric_value", T.DoubleType(), True),
#         T.StructField("threshold", T.DoubleType(), True),
#         T.StructField("status", T.StringType(), False),
#         T.StructField("details", T.StringType(), True),
#     ])

#     df = spark.createDataFrame(rows,schema=schema).withColumn("checked_at", F.current_timestamp())
#     return df

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 14, Finished, Available, Finished, False)

In [13]:
# # Write the data quality results DataFrame to a Delta table in append mode as a persistent report
# def write_dq_report(dq_df, table_name="gold_dq_report"):
#     (
#         dq_df
#         .select("run_id","layer","dataset","rule_name","rule_type", "column_name", "metric_value", "threshold","status", "checked_at", "details")
#         .write
#         .mode("append")
#         .option("mergeSchema", "true")
#         .format("delta")
#         .saveAsTable(table_name)
#     )

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 15, Finished, Available, Finished, False)